# Notebook 08 — Voice Agent (Sequential Pipeline)

Closing Phase 3. We wire Whisper into a complete voice agent: **STT → LLM (with tool calls) → TTS**, and measure the **latency budget** that determines whether the agent feels broken to a human (S3 §6.2).

## The architecture (S3 §6.1)

```
audio in
  ↓
VAD → detect speech
  ↓
STT (faster-whisper) → text
  ↓
LLM (GPT-4o, tool calls)
  ↓
TTS (OpenAI tts-1) → audio chunks
  ↓
audio out
```

Each stage has its own latency. The **800 ms** rule (§6.2): if total perceived latency exceeds ~800 ms, the conversation feels broken. Below that, it feels natural.

Engineering implication: **everything streams.** STT emits partial transcripts; LLM streams tokens; TTS synthesizes audio chunks as tokens arrive. We're going to build the synchronous version first (clearer to read), measure it, and then show the streaming version to see where the wins come from.

## What this notebook covers

1. Set up STT + LLM (with a mock weather tool) + TTS.
2. Synchronous pipeline with per-stage timing on three queries (EN, ZH, code-switched).
3. Streaming version — sentence-level TTS as LLM emits tokens. Measure time-to-first-audio.
4. Compare to speech-to-speech architectures.

## 1. Setup

In [ ]:
import time, json, re
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from faster_whisper import WhisperModel

load_dotenv(dotenv_path="../.env")
client = OpenAI()

AUDIO_DIR = Path("../data/audio_samples/agent")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

asr = WhisperModel("large-v3", device="cpu", compute_type="int8")
print("asr loaded")

In [ ]:
# Mock tool — production would call a real API; the point is the tool-use pattern.
WEATHER_DB = {
    "tokyo":   {"temp_c": 18, "condition": "partly cloudy"},
    "beijing": {"temp_c":  5, "condition": "clear"},
    "london":  {"temp_c": 11, "condition": "rainy"},
    "taipei":  {"temp_c": 23, "condition": "humid"},
}

def get_weather(city: str) -> dict:
    return WEATHER_DB.get(city.strip().lower(), {"error": f"no data for {city}"})

TOOL_SCHEMA = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather for a city",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
}]

In [ ]:
# Test queries — we TTS them to make 'voice input' for the lab.
def make_input_audio(name: str, text: str) -> Path:
    p = AUDIO_DIR / f"in_{name}.mp3"
    if p.exists():
        return p
    client.audio.speech.create(model="tts-1", voice="nova", input=text).stream_to_file(str(p))
    return p

TEST_QUERIES = [
    ("en",     "What's the weather like in Tokyo today?"),
    ("zh",     "今天东京的天气怎么样？"),
    ("mixed",  "今天 Tokyo 的 weather 怎么样？"),
]

input_paths = {name: make_input_audio(name, q) for name, q in TEST_QUERIES}
print(f"prepared {len(input_paths)} input recordings")

## 2. Synchronous pipeline with per-stage timing

The simplest version. Wait for each stage to finish, then start the next. We'll measure how much we lose by *not* streaming.

In [ ]:
SYSTEM_PROMPT = (
    "You are a concise voice assistant. Answer in one or two short sentences. "
    "Match the user's language (English, Chinese, or mixed). Use tools when needed."
)

def stage_stt(audio_path: Path) -> tuple[str, float]:
    t0 = time.time()
    segments, info = asr.transcribe(str(audio_path), vad_filter=True)
    text = " ".join(s.text.strip() for s in segments)
    return text, time.time() - t0

def stage_llm(user_text: str) -> tuple[str, float]:
    t0 = time.time()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_text},
    ]
    r = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=TOOL_SCHEMA,
    )
    msg = r.choices[0].message

    # If the model called a tool, run it and round-trip.
    if msg.tool_calls:
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments)
            result = get_weather(**args) if call.function.name == "get_weather" else {"error": "unknown tool"}
            messages.append({
                "role": "tool", "tool_call_id": call.id, "content": json.dumps(result),
            })
        r2 = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
        text = r2.choices[0].message.content
    else:
        text = msg.content
    return text, time.time() - t0

def stage_tts(text: str, out_path: Path) -> float:
    t0 = time.time()
    audio = client.audio.speech.create(model="tts-1", voice="nova", input=text)
    audio.stream_to_file(str(out_path))
    return time.time() - t0

In [ ]:
from IPython.display import Audio, display

def run_sync(name: str, audio_path: Path):
    out_path = AUDIO_DIR / f"out_sync_{name}.mp3"
    transcript, dt_stt = stage_stt(audio_path)
    answer,     dt_llm = stage_llm(transcript)
    dt_tts             = stage_tts(answer, out_path)
    total = dt_stt + dt_llm + dt_tts
    print(f"\n=== {name} ===")
    print(f"  user said:  {transcript.strip()}")
    print(f"  agent said: {answer}")
    print(f"  STT {dt_stt*1000:.0f} ms  +  LLM {dt_llm*1000:.0f} ms  +  TTS {dt_tts*1000:.0f} ms  =  {total*1000:.0f} ms total")
    display(Audio(str(out_path)))
    return {"stt": dt_stt, "llm": dt_llm, "tts": dt_tts, "total": total}

sync_results = {name: run_sync(name, path) for name, path in input_paths.items()}

## 3. Latency budget analysis

The 800 ms rule (S3 §6.2): exceed it and the agent feels broken. Where are we?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = list(sync_results)
stt = [sync_results[n]["stt"] * 1000 for n in names]
llm = [sync_results[n]["llm"] * 1000 for n in names]
tts = [sync_results[n]["tts"] * 1000 for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(names))
ax.bar(x, stt,                    label="STT")
ax.bar(x, llm, bottom=stt,        label="LLM (incl. tool round-trip)")
ax.bar(x, tts, bottom=np.array(stt) + np.array(llm), label="TTS")
ax.axhline(800, color="red", linestyle="--", label="800 ms — broken-feeling threshold")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("ms")
ax.set_title("Synchronous pipeline — where the latency goes")
ax.legend()
plt.tight_layout(); plt.show()

Most likely outcome: total is well over 800 ms, dominated by **STT** (Whisper-large on CPU is slow) and **LLM** (especially when there's a tool round-trip — that's two API calls in series).

## 4. Streaming the LLM → sentence-by-sentence TTS

The streaming insight: **the user doesn't need to hear the last sentence to start hearing the first**. As soon as the LLM finishes the first sentence, fire it off to TTS in parallel with the LLM still generating. *Time to first audio* — not total time — is what determines the feel.

Pattern:
1. Stream LLM tokens.
2. Buffer until a sentence boundary (`.`, `?`, `!`, `。`, `？`, `！`).
3. Send each completed sentence to TTS as it's emitted.
4. Stitch the audio chunks together at playback time.

Below: a minimal implementation. We don't bother stitching — we just measure **time-to-first-audio-byte**, which is what would land in the user's headphones first.

In [ ]:
SENTENCE_END = re.compile(r"[.!?。！？]\s*")

def stream_llm_to_sentences(user_text: str):
    """Yield (sentence, time_since_start) as the LLM emits sentence-bounded text."""
    t0 = time.time()
    stream = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT + " (No tool calls in this run — keep it text-only for streaming demo.)"},
            {"role": "user",   "content": user_text},
        ],
        stream=True,
    )
    buf = ""
    for chunk in stream:
        delta = chunk.choices[0].delta.content or ""
        buf += delta
        # Yield each completed sentence as soon as it appears
        while True:
            m = SENTENCE_END.search(buf)
            if not m:
                break
            cut = m.end()
            sentence, buf = buf[:cut].strip(), buf[cut:]
            if sentence:
                yield sentence, time.time() - t0
    if buf.strip():
        yield buf.strip(), time.time() - t0


def first_audio_byte_streaming(user_text: str) -> dict:
    """Return timings for the streaming pipeline (post-STT)."""
    t0 = time.time()
    first_sentence_time = None
    first_audio_time = None
    for sentence, t_sent in stream_llm_to_sentences(user_text):
        if first_sentence_time is None:
            first_sentence_time = t_sent
            with client.audio.speech.with_streaming_response.create(
                model="tts-1", voice="nova", input=sentence,
            ) as resp:
                for _ in resp.iter_bytes(chunk_size=1024):
                    first_audio_time = time.time() - t0
                    break
            break   # we only need to time the first sentence for this measurement
    return {
        "time_to_first_sentence": first_sentence_time,
        "time_to_first_audio":   first_audio_time,
    }

for name, q in TEST_QUERIES:
    timings = first_audio_byte_streaming(q)
    print(f"{name}: first sentence={timings['time_to_first_sentence']*1000:.0f} ms  →  first audio byte={timings['time_to_first_audio']*1000:.0f} ms")

Compare these numbers to the synchronous totals from §3. The streaming version's time-to-first-audio is typically **2–4× faster than the sync total**, because:

- LLM streaming hands you the first sentence before it's done generating the whole answer.
- TTS streaming hands you the first audio byte before it's done synthesizing the whole sentence.
- These two streams pipeline: while TTS synthesizes sentence 1, LLM is generating sentence 2.

**This is the "everything streams" rule (S3 §6.2).** Sequential pipelines aren't slow because the stages are slow individually — they're slow because we *wait* between them.

## 5. The other architecture — speech-to-speech

GPT-4o Realtime, Gemini Live, and Moshi are **speech-to-speech** models — one model takes audio in, emits audio out, no STT/TTS hops. Tradeoffs (S3 §6.4):

| | Sequential (this notebook) | Speech-to-speech |
|---|---|---|
| Latency | 600–1200 ms (with streaming) | 300–500 ms |
| Voice expressiveness | Limited — text loses prosody | High — preserves emotion, pauses |
| Tool calling | Mature (LLM step is text) | Limited (still maturing) |
| Debuggability | High — log every text step | Lower — audio-in / audio-out is a black box |
| Provider lock-in | Low — mix STT, LLM, TTS vendors | High |

Pick by what matters: **tool calls + RAG + observability → sequential**. **Natural feel + can tolerate lock-in → speech-to-speech.**

## What we built

A complete sequential voice agent: faster-whisper STT (with VAD), GPT-4o (with tool calls), OpenAI TTS — both in a synchronous version (clear) and a streaming version (fast). With per-stage timing measurements you can inspect to find the bottleneck in your own deployment.

Production checklist (S3 §10.1):
- ☐ Use `faster-whisper` (or Distil-Whisper) — never `openai-whisper` direct.
- ☐ Cap user audio length (>30 s should be split or rejected).
- ☐ Stream every stage; never wait for full completion.
- ☐ Log per-stage latency separately. Total latency tells you the agent is broken; per-stage latency tells you which stage to fix.
- ☐ Add a confidence gate from word-level probabilities (nb06) — refuse to act on low-confidence transcripts.

**End of Phase 3.** Phase 4 is video — same idea: split into images-over-time + audio, index both, search across.

Onward: [Notebook 09 — Video Indexing](09_video_indexing.ipynb).